In [10]:
!py -m pip install pandas openpyxl xlrd -q


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

### Convert xls to csv data

In [4]:
df = pd.read_excel("D:\mycode\Fintech-agent\data\macro\Trade balance.xlsx", sheet_name="Query Results")
df.columns = df.columns.str.strip()

df = df.rename(columns={
    "Year": "year",
    "Month": "month",
    df.columns[-1]: "trade_balance"
})

df = df.dropna(subset=["year", "month"])

df["year"] = df["year"].astype(int)
df["month"] = df["month"].astype(int)
df_q = df[df["month"].isin([3, 6, 9, 12])].copy()

df_q["date"] = pd.to_datetime(
    df_q["year"].astype(str) + "-" +
    df_q["month"].astype(str) + "-01"
) + pd.offsets.MonthEnd(0)


df_q = df_q[["date", "trade_balance"]].sort_values("date")
output_path = r"data/macro/Trade balance.csv"
df_q.to_csv(output_path, index=False)

print(f"Converted successfully → {output_path}")

Converted successfully → data/macro/Trade balance.csv


In [ ]:
df = pd.read_excel("data/macro/gscpi_data.xls", sheet_name = "GSCPI Monthly Data")
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["Date"])
df.to_csv("data/macro/gscpi.csv", index=False)
print(f"Converted!")


### Merge CSVs

In [21]:
def to_quarter_end(series: pd.Series) -> pd.Series:
    return  series.dt.to_period("Q").dt.to_timestamp("Q") 

In [22]:
inflation = pd.read_csv(
    r"D:\mycode\Fintech-agent\data\macro\Treasury and Inflation.csv",
    parse_dates=["caldt"],
    dayfirst=True       
)


inflation["date"] = to_quarter_end(inflation["caldt"])
inflation = inflation.drop(columns=["caldt"])

inflation = (
    inflation
    .groupby("date")
    .last()
    .reset_index()
)

wti = pd.read_csv(
    r"D:\mycode\Fintech-agent\data\macro\Crude oil price.csv",
    skiprows=4,
    parse_dates=["Month"]
)
wti = wti.rename(columns={
    "Cushing OK WTI Spot Price FOB Dollars per Barrel": "wti_price"
})


wti["wti_price"] = pd.to_numeric(wti["wti_price"], errors="coerce")
wti["date"] = to_quarter_end(wti["Month"])
wti = wti.sort_values("date")

wti = (
    wti.groupby("date")
    .agg(wti_price=("wti_price", "mean"))   
    .reset_index()
)

merged = pd.merge(inflation, wti, on="date", how="left")
merged = merged.sort_values("date").reset_index(drop=True)

In [23]:
fed = pd.read_csv(
    r"D:\mycode\Fintech-agent\data\macro\Federal Funds Effective(Interest) Rate.csv",
    parse_dates=["observation_date"]
)
fed = fed.rename(columns={"observation_date": "date", "FEDFUNDS": "fedfunds"})
fed["fedfunds"] = pd.to_numeric(fed["fedfunds"], errors="coerce")
 
fed["date"] = to_quarter_end(fed["date"])
 
fed = (
    fed.groupby("date")
    .agg(fedfunds=("fedfunds", "mean"))
    .reset_index()
)
merged = pd.merge(merged, fed, on="date", how="left")
merged = merged.sort_values("date").reset_index(drop=True)

In [24]:

nfci = pd.read_csv(
    r"D:\mycode\Fintech-agent\data\macro\financial conditions index.csv",
    parse_dates=False
)
 
nfci = nfci.rename(columns={
    "Friday_of_Week":        "date",
    "NFCI":                  "nfci",
    "ANFCI":                 "anfci",
    "Risk":                  "nfci_risk",
    "Credit":                "nfci_credit",
    "Leverage":              "nfci_leverage",
    "Nonfinancial_Leverage": "nfci_nonfinancial_leverage"
})
 
nfci["date"] = pd.to_datetime(nfci["date"], format="%m/%d/%Y")
nfci["date"] = to_quarter_end(nfci["date"])
 
nfci = (
    nfci.groupby("date")
    .agg(
        nfci=("nfci", "mean"),
        anfci=("anfci", "mean"),
        nfci_risk=("nfci_risk", "mean"),
        nfci_credit=("nfci_credit", "mean"),
        nfci_leverage=("nfci_leverage", "mean"),
        nfci_nonfinancial_leverage=("nfci_nonfinancial_leverage", "mean")
    )
    .reset_index()
)
 
merged = pd.merge(merged, nfci, on="date", how="left")
merged = merged.sort_values("date").reset_index(drop=True)
 

In [25]:
gdp = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\GDP.csv", parse_dates=["observation_date"])
gdp = gdp.rename(columns={"observation_date": "date", "GDP": "gdp"})
gdp["gdp"] = pd.to_numeric(gdp["gdp"], errors="coerce")
gdp["date"] = to_quarter_end(gdp["date"])
gdp = gdp.groupby("date").agg(gdp=("gdp", "last")).reset_index()
merged = pd.merge(merged, gdp, on="date", how="left")
 
ppi = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\PPI.csv", parse_dates=["observation_date"])
ppi = ppi.rename(columns={"observation_date": "date", "PPIACO": "ppi"})
ppi["ppi"] = pd.to_numeric(ppi["ppi"], errors="coerce")
ppi["date"] = to_quarter_end(ppi["date"])
ppi = ppi.groupby("date").agg(ppi=("ppi", "mean")).reset_index()
merged = pd.merge(merged, ppi, on="date", how="left")
 
tb = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\Trade balance.csv", parse_dates=["date"])
tb["trade_balance"] = pd.to_numeric(tb["trade_balance"], errors="coerce")
tb["date"] = to_quarter_end(tb["date"])
tb = tb.groupby("date").agg(trade_balance=("trade_balance", "sum")).reset_index()
merged = pd.merge(merged, tb, on="date", how="left")
 
unrate = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\UNRate.csv")
unrate.columns = ["date", "unrate"]
unrate["date"] = pd.to_datetime(unrate["date"], format="%m/%d/%Y")
unrate["unrate"] = pd.to_numeric(unrate["unrate"], errors="coerce")
unrate["date"] = to_quarter_end(unrate["date"])
unrate = unrate.groupby("date").agg(unrate=("unrate", "mean")).reset_index()
merged = pd.merge(merged, unrate, on="date", how="left")
 
merged = merged.sort_values("date").reset_index(drop=True)

In [ ]:
usd = pd.read_csv(r"D:\mycode\Fintech-agent\data\macro\US Dollar Index Historical Data.csv")
usd = usd[["Date", "Price"]].rename(columns={"Date": "date", "Price": "usd_index"})
usd["date"] = pd.to_datetime(usd["date"], format="%m-%d-%Y")
usd["usd_index"] = pd.to_numeric(usd["usd_index"], errors="coerce")
usd["date"] = to_quarter_end(usd["date"])
usd = usd.sort_values("date")
usd = usd.groupby("date").agg(usd_index=("usd_index", "mean")).reset_index()
merged = pd.merge(merged, usd, on="date", how="left")
merged = merged.sort_values("date").reset_index(drop=True)
 

In [ ]:
merged = merged[
    (merged["date"] >= "1990-01-01") &
    (merged["date"] <= "2024-12-31")
].reset_index(drop=True)
 

In [ ]:
housing = pd.read_csv("D:\mycode\Fintech-agent\data\macro\housing.csv", encoding="utf-16",sep="\t")     

housing["date"] = pd.to_datetime(housing["Month of Period End"], format="%B %Y")
housing["date"] = housing["date"].dt.to_period("Q").dt.to_timestamp("Q")
housing = housing.drop(columns=["Region", "Month of Period End"])
 
mom_yoy_cols = [c for c in housing.columns if "MoM" in c or "YoY" in c]
housing = housing.drop(columns=mom_yoy_cols)
def clean_column(series):
    """Remove $, K, % symbols and convert to float"""
    if series.dtype == object:
        series = (series
                  .str.replace("$", "", regex=False)
                  .str.replace("K", "", regex=False)
                  .str.replace("%", "", regex=False)
                  .str.replace(",", "", regex=False)
                  .str.strip())
        return pd.to_numeric(series, errors="coerce")
    return series
 
housing["Median Sale Price"] = (
    housing["Median Sale Price"]
    .str.replace("$", "", regex=False)
    .str.replace("K", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors="coerce") * 1000
)

for col in housing.columns:
    if col != "date" and col != "Median Sale Price":
        housing[col] = clean_column(housing[col])

housing_quarterly = housing.groupby("date").mean().reset_index()
merged["date"] = pd.to_datetime(merged["date"])
merged["date"] = merged["date"].dt.to_period("Q").dt.to_timestamp("Q")
merged = pd.merge(merged, housing_quarterly, on="date", how="inner")
 

In [31]:
merged.to_csv("D:\mycode\Fintech-agent\data\macro\combined_macro_data.csv", index=False)
print(f"Saved: data/macro/combined_macro_data.csv")
print(f"Shape: {merged.shape[0]} rows × {merged.shape[1]} columns")
print(f"Date range: {merged['date'].min().date()} → {merged['date'].max().date()}")
print(f"\nColumns:\n{list(merged.columns)}")

Saved: data/macro/combined_macro_data.csv
Shape: 140 rows × 24 columns
Date range: 1990-03-31 → 2024-12-31

Columns:
['date', 'b30ret', 'b20ret', 'b10ret', 'b7ret', 'b5ret', 'b2ret', 'b1ret', 't90ret', 't30ret', 'cpiret', 'wti_price', 'fedfunds', 'nfci', 'anfci', 'nfci_risk', 'nfci_credit', 'nfci_leverage', 'nfci_nonfinancial_leverage', 'gdp', 'ppi', 'trade_balance', 'unrate', 'usd_index']


In [30]:
print(merged.isnull().sum())

date                          0
b30ret                        0
b20ret                        0
b10ret                        0
b7ret                         0
b5ret                         0
b2ret                         0
b1ret                         0
t90ret                        0
t30ret                        0
cpiret                        0
wti_price                     0
fedfunds                      0
nfci                          0
anfci                         0
nfci_risk                     0
nfci_credit                   0
nfci_leverage                 0
nfci_nonfinancial_leverage    0
gdp                           0
ppi                           0
trade_balance                 0
unrate                        0
usd_index                     0
dtype: int64
